Feature Extraction Folder — Complete Guide

  This is an audio anomaly detection pipeline that converts .wav files into numpy arrays for use in machine learning
  models. Here's how everything fits together.

  ---
  The Big Picture


      .wav file
      │

      ├─ remove background noise (noisereduce)
      │

      ├─→ [Path A] Mel Spectrogram image (128×313×3 numpy array)  ← for CNN/image models
      │

      └─→ [Path B] Audio feature vector (8-element numpy array)   ← for classical ML / tabular models

  ---
  File-by-File Explanation

  features.py — The Core Engine

  This is where all the real work happens. Six functions:

  ---
  create_master_mask(normal_files)

  Takes 20 normal .wav files and extracts 0.5 seconds from each, concatenates them into one long array — a "noise
  profile" of what the machine sounds like when healthy.

  master_noise = create_master_mask(normal_files)
   Result: a 1D numpy array of background noise samples

  ---
  remove_background_noise(y, sr, master_noise)

  Uses the noisereduce library to subtract the master noise profile from a raw audio signal. prop_decrease=1.0 means
  full noise subtraction.

  y_clean = remove_background_noise(y, sr, master_noise)

  ---
  get_normal_baseline(normal_files)

  Looks at 20 normal files and fits 3 MinMaxScaler objects on them — one each for mel, delta, delta². These scalers
  define what "normal range" looks like, so every file gets normalized relative to the normal baseline.

  scaler_mel, scaler_delta, scaler_delta2 = get_normal_baseline(normal_files)
   Returns 3 fitted sklearn scalers (saved to .pkl files for reuse)

  ---
  process_log_mel_spectrogram(y, sr, scaler_mel=None) → shape (128, 313)

  Converts raw audio to a log-mel spectrogram:

  1. librosa.feature.melspectrogram(...) — applies 128 mel filter banks
  2. librosa.power_to_db(...) — converts to decibels (log scale)
  3. Pad/trim to exactly 313 time frames (= ~10 seconds at 16kHz with hop=512)
  4. MinMax scale to [0, 1]

  Result: a 2D image (128 mels × 313 time frames) normalized to [0,1].

  ---
  process_delta(series, scaler, order=1 or 2) → same shape as input

  Computes the temporal derivative of the spectrogram:
  - order=1 (delta): how fast each frequency bin is changing over time
  - order=2 (delta²): acceleration — how fast the change is changing

  These are standard speech/audio features. Combined with the mel, they give the model motion information, not just
  static frequency content.

  ---
  process_file(file_path, scaler_mel, scaler_delta, scaler_delta2, master_noise, no_mel=False)

  The main entry point. Does everything in sequence:

   With no_mel=False (default):
  mel_image, audio_features = process_file(...)
   mel_image shape:     (128, 313, 3)  ← stacked: [mel, delta, delta²]
   audio_features shape: (8,)          ← [ZCR, mean_amp, amp_var, pitch, centroid, rolloff, mfcc1, mfcc2]

   With no_mel=True:
  audio_features = process_file(..., no_mel=True)
   Returns only the 8-element vector

  The (128, 313, 3) array is like a 3-channel image — mel is the red channel, delta is green, delta² is blue. CNNs can
   consume this directly.

  ---
  extract_audio_features(y, sr) → shape (8,)

  Extracts 8 hand-crafted features and returns a float32 array:

  ┌───────┬──────────────────────────┬───────────────────────────────────────────────────────────────────────┐
  │ Index │         Feature          │                           What it captures                            │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 0     │ ZCR (Zero Crossing Rate)   │ How often the signal crosses zero — high for noisy/slider sounds      │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 1     │ Mean Amplitude (RMS)     │ Overall loudness                                                      │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 2     │ Amplitude Variance       │ How much loudness fluctuates                                          │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 3     │ Median Pitch (f0)        │ Fundamental frequency via pYIN algorithm                              │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 4     │ Spectral Centroid        │ "Center of mass" of frequencies — high = brighter/higher freq content │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 5     │ Spectral Rolloff         │ Frequency below which 85% of energy lies                              │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 6     │ MFCC 1                   │ Timbre/texture of sound                                               │
  ├───────┼──────────────────────────┼───────────────────────────────────────────────────────────────────────┤
  │ 7     │ MFCC 2                   │ Timbre/texture of sound                                               │
  └───────┴──────────────────────────┴───────────────────────────────────────────────────────────────────────┘

  ---
  prepare_dataset.py — Batch Processing (Training Data)

  Run this once when building a dataset. It:
  1. Scans a folder of .wav files recursively
  2. Builds the noise mask and fits scalers from the last 20 files (assumed normal)
  3. Saves scalers to .pkl files
  4. Processes every .wav in parallel using ProcessPoolExecutor
  5. Saves two .npy files per audio file: filename_mel_data.npy and filename_audio_features.npy

  raw_data/valve/id_00/normal/00000000.wav
      → processed_features/valve/id_00/normal/00000000_mel_data.npy     (128×313×3)
      → processed_features/valve/id_00/normal/00000000_audio_features.npy (8,)

  How to run it: It reads config.INPUT_DIRS, config.OUTPUT_DIRS, config.SCALER_DIRS (a config.py you need to have at
  the project root).

  ---
  inference.py — Single File at Runtime

  Used when you have a new audio file and want features from it (e.g., during live inference). It loads the pre-saved
  scalers and runs process_file.

  from feature_extraction.inference import process_incoming_audio
  from pathlib import Path

   Returns (mel_image, audio_features) or just audio_features if no_mel=True
  result = process_incoming_audio(Path("my_recording.wav"))
  mel_image, audio_features = result
   mel_image: numpy array (128, 313, 3)
   audio_features: numpy array (8,)

  Prerequisite: The scaler .pkl files must exist at config.SCALER_DIRS[0] — they're generated by prepare_dataset.py.

  ---
  visualise.py — Debugging / Exploration

  Three utility functions for plotting:

  - plot_mel_spectrogram(*file_paths) — plots one or more mel spectrograms side by side
  - plot_difference_mel_spectrogram(file1, file2, ...) — subtracts two spectrograms to highlight differences between
  normal and abnormal
  - convert_to_audio(norm_mel, output) — reconstructs audio from a mel spectrogram (for sanity-checking)

  ---
  preprocess.py — Old/Standalone Version

  This is an older standalone script with its own hardcoded paths. It duplicates a lot of features.py logic. You can
  mostly ignore it — features.py + prepare_dataset.py is the current, cleaner version.

  ---
  How to Use It — Step by Step

  Step 1: Set Up config.py

  Create a config.py at the project root (where feature_extraction/ lives):

  from pathlib import Path

  SAMPLING_RATE = 16000
  N_FFT = 2048
  HOP_LENGTH = 512
  N_MELS = 128
  FIXED_WIDTH = 313

  INPUT_DIRS  = [Path("raw_data/valve/id_00")]
  OUTPUT_DIRS = [Path("processed_features/valve/id_00")]
  SCALER_DIRS = [Path("scalers/valve/id_00")]

  Step 2: Organize Your .wav Files

  raw_data/valve/id_00/
      normal/
          00000000.wav
          00000001.wav
          ...
      abnormal/
          00000000.wav
          ...

  Step 3: Run prepare_dataset.py (One Time)

  python feature_extraction/prepare_dataset.py

  This generates .pkl scalers and .npy feature files.